In [1]:
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from astropy.wcs import WCS
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_sun
import astropy.units as u
from scipy.signal import savgol_filter
from datetime import datetime
import shutil
import os
import glob
import re


In [7]:


class RadioPipelineGRAD300:
    def __init__(self, base_path, output_base="output_nametarget"):
        """
        Initialise le pipeline pour les données GRAD-300
        """
        self.base_path = base_path
        self.output_base = output_base
        self.ignored_files = []  # Liste pour stocker les fichiers ignorés
        self.processed_files = []  # Liste pour stocker les fichiers traités
        self.error_files = []  # Liste pour stocker les fichiers en erreur
        
        # Paramètres pour le TPI
        self.window_length = 7
        self.polyorder = 2
        
        # Paramètres pour le spectre
        self.basefreq = 1.398e9
        self.bandwidth = 62.5e6
        self.nchan = 1024
        
        # Localisation de l'observatoire
        self.location = EarthLocation(
            lat=43.933 * u.deg,
            lon=5.7153 * u.deg,
            height=654.8 * u.m
        )
        
        # Pattern pour parser les noms de fichiers
        self.file_pattern = r'(\d{8})-(\d{6})_([A-Z]+)-([A-Z0-9]+)-([A-Z0-9]+)_\d+#_\d+#\.fits'
    
    def is_valid_spectrum(self, table):
        """
        Vérifie si un fichier est un vrai spectre avec les colonnes attendues
        """
        if not hasattr(table, 'dtype') or table.dtype.names is None:
            return False
        
        cols = table.dtype.names
        required_cols = ['LEFT_POL', 'RIGHT_POL', 'STATUS']
        
        # Vérifier la présence des colonnes requises
        has_required = all(col in cols for col in required_cols)
        
        # Vérifier que ce n'est PAS un fichier de tracking
        is_tracking = 'Azimuth' in cols and 'Elevation' in cols
        
        # Vérifier qu'il y a des données ON/OFF
        has_onoff = False
        if has_required and 'STATUS' in cols:
            has_on = np.any(table['STATUS'] == 'on')
            has_off = np.any(table['STATUS'] == 'off')
            has_onoff = has_on and has_off
        
        return has_required and not is_tracking and has_onoff
    
    def is_valid_tpi(self, table):
        """
        Vérifie si un fichier est un vrai TPI
        """
        if not hasattr(table, 'dtype') or table.dtype.names is None:
            return False
        
        cols = table.dtype.names
        required_cols = ['Azimuth', 'Elevation', 'JD']
        
        return all(col in cols for col in required_cols)
    
    def find_observation_files(self, observation_dir):
        """
        Trouve et regroupe les fichiers par cible dans le dossier FITS
        """
        fits_dir = os.path.join(observation_dir, "FITS")
        
        if not os.path.exists(fits_dir):
            print(f"  ⚠️  Dossier FITS non trouvé dans {observation_dir}")
            return {}
        
        files_by_target = {}
        all_fits_files = glob.glob(os.path.join(fits_dir, "*.fits"))
        
        print(f"  📁 {len(all_fits_files)} fichiers trouvés dans {fits_dir}")
        
        for file_path in all_fits_files:
            filename = os.path.basename(file_path)
            match = re.match(self.file_pattern, filename)
            
            if match:
                date = match.group(1)
                time = match.group(2)
                file_type = match.group(3)  # IMAGE, TPI, SPECTRUM
                instrument = match.group(4)
                target = match.group(5)
                
                # Initialiser la structure pour cette cible si nécessaire
                if target not in files_by_target:
                    files_by_target[target] = {
                        'image': [], 
                        'tpi': [], 
                        'spectrum': [],
                        'ignored': []  # Pour les fichiers ignorés
                    }
                
                file_info = {
                    'path': file_path,
                    'time': time,
                    'filename': filename,
                    'datetime': f"{date}-{time}",
                    'type_annonce': file_type
                }
                
                # Pour les fichiers SPECTRUM, on va vérifier le contenu plus tard
                # On les met dans spectrum pour l'instant, on vérifiera au traitement
                if file_type == "SPECTRUM":
                    files_by_target[target]['spectrum'].append(file_info)
                elif file_type == "TPI":
                    files_by_target[target]['tpi'].append(file_info)
                elif file_type == "IMAGE":
                    files_by_target[target]['image'].append(file_info)
                else:
                    files_by_target[target]['ignored'].append({
                        **file_info,
                        'reason': f"Type non reconnu: {file_type}"
                    })
            else:
                # Fichier ne correspondant pas au pattern
                target = "UNKNOWN"
                if target not in files_by_target:
                    files_by_target[target] = {'image': [], 'tpi': [], 'spectrum': [], 'ignored': []}
                
                files_by_target[target]['ignored'].append({
                    'path': file_path,
                    'filename': filename,
                    'reason': "Nom de fichier non conforme au pattern"
                })
        
        return files_by_target
    
    def process_spectrum_unprocessed(self, spectrum_file_info, target, obs_date):
        """
        Traite les données de spectre en mode unprocessed (brut)
        Ne traite que les vrais spectres, ignore les autres
        """
        file_path = spectrum_file_info['path']
        filename = os.path.basename(file_path)
        
        print(f"  📊 Vérification spectre: {filename}")
        
        try:
            with fits.open(file_path, ignore_missing_end=True) as hdul:
                # Chercher les données
                if len(hdul) > 1 and hdul[1].data is not None:
                    table = hdul[1].data
                else:
                    table = hdul[0].data
                
                # Vérifier si c'est un vrai spectre
                if not self.is_valid_spectrum(table):
                    reason = "Pas un vrai spectre (manque LEFT_POL/RIGHT_POL/STATUS ou contient Az/El)"
                    print(f"     ⚠️  Ignoré: {reason}")
                    self.ignored_files.append({
                        'file': filename,
                        'target': target,
                        'obs_date': obs_date,
                        'type': 'SPECTRUM',
                        'reason': reason,
                        'columns': list(table.dtype.names) if hasattr(table, 'dtype') and table.dtype.names else []
                    })
                    return None
                
                print(f"     ✅ Vrai spectre détecté")
                self.processed_files.append({
                    'file': filename,
                    'target': target,
                    'obs_date': obs_date,
                    'type': 'SPECTRUM'
                })
                
                # Extraire les données
                result = {
                    'raw_data': table,
                    'filename': filename,
                    'columns': list(table.dtype.names)
                }
                
                # Extraire ON/OFF
                on_rows = table[table['STATUS'] == 'on']
                off_rows = table[table['STATUS'] == 'off']
                cal_rows = table[table['STATUS'] == 'cal']
                
                if len(on_rows) > 0:
                    result['on'] = on_rows[0]
                if len(off_rows) > 0:
                    result['off'] = off_rows[0]
                if len(cal_rows) > 0:
                    result['cal'] = cal_rows[0]
                
                # Combiner les polarisations
                if len(on_rows) > 0:
                    result['spec_on'] = on_rows[0]['LEFT_POL'] + on_rows[0]['RIGHT_POL']
                if len(off_rows) > 0:
                    result['spec_off'] = off_rows[0]['LEFT_POL'] + on_rows[0]['RIGHT_POL']
                
                # Créer un tableau de fréquences
                freq = self.basefreq + np.linspace(0, self.bandwidth, self.nchan)
                result['freq'] = freq
                
                return result
                
        except Exception as e:
            print(f"  ❌ Erreur lecture: {e}")
            self.error_files.append({
                'file': filename,
                'target': target,
                'obs_date': obs_date,
                'type': 'SPECTRUM',
                'error': str(e)
            })
            return None
    
    def process_tpi(self, tpi_file_info, target, obs_date):
        """
        Traite les données TPI (tracking) avec tout le traitement complet
        """
        file_path = tpi_file_info['path']
        filename = os.path.basename(file_path)
        
        print(f"  🎯 Traitement TPI: {filename}")
        
        try:
            with fits.open(file_path) as hdultpi:
                if len(hdultpi) > 1 and hdultpi[1].data is not None:
                    table_tpi = hdultpi[1].data
                else:
                    table_tpi = hdultpi[0].data
                
                # Vérifier si c'est un vrai TPI
                if not self.is_valid_tpi(table_tpi):
                    reason = "Pas un vrai TPI (manque Azimuth/Elevation/JD)"
                    print(f"     ⚠️  Ignoré: {reason}")
                    self.ignored_files.append({
                        'file': filename,
                        'target': target,
                        'obs_date': obs_date,
                        'type': 'TPI',
                        'reason': reason,
                        'columns': list(table_tpi.dtype.names) if hasattr(table_tpi, 'dtype') and table_tpi.dtype.names else []
                    })
                    return None
                
                self.processed_files.append({
                    'file': filename,
                    'target': target,
                    'obs_date': obs_date,
                    'type': 'TPI'
                })
                
                # ===== TRAITEMENT TPI COMPLET =====
                
                # Filtrage temporel (ignorer les premières secondes)
                try:
                    t = (table_tpi['JD'] - table_tpi['JD'][0]) * 86400
                    mask_time = t > 1
                    table_tpi = table_tpi[mask_time]
                except:
                    print("     ⚠️  Pas de colonne JD, pas de filtrage temporel")
                
                # 👉 Vérifier que table_tpi n'est pas vide après filtrage
                if len(table_tpi) == 0:
                    print("     ⚠️  Aucune donnée après filtrage temporel")
                    return {'raw': table_tpi, 'clean': table_tpi, 'has_altaz': False}
                
                # Préparation des données
                try:
                    t_cut = (table_tpi['JD'] - table_tpi['JD'][0]) * 86400
                    dt = np.median(np.diff(t_cut))
                except:
                    dt = 1.0  # Valeur par défaut
                    t_cut = np.arange(len(table_tpi))
                
                # Conversion Alt/Az -> RA/Dec si les colonnes existent
                has_altaz = ('Azimuth' in table_tpi.dtype.names and 
                            'Elevation' in table_tpi.dtype.names)
                
                if has_altaz:
                    az = table_tpi["Azimuth"]
                    el = table_tpi["Elevation"]
                    
                    # Gestion du temps
                    if 'JD' in table_tpi.dtype.names:
                        jd = table_tpi['JD']
                        time = Time(jd, format='jd')
                    else:
                        # Utiliser l'index comme temps si pas de JD
                        time = Time.now() + (t_cut * u.s)
                    
                    altaz = AltAz(
                        az=az * u.deg,
                        alt=el * u.deg,
                        location=self.location,
                        obstime=time
                    )
                    
                    sky = SkyCoord(altaz)
                    ra = sky.icrs.ra.deg
                    dec = sky.icrs.dec.deg
                    
                    # Filtrage sigma-clip
                    ra_mask = self.sigma_clip_mask_robust(ra)
                    dec_mask = self.sigma_clip_mask_robust(dec)
                    mask_combined = ra_mask & dec_mask
                    
                    # 👉 Vérifier qu'il reste des points après sigma-clip
                    if np.sum(mask_combined) == 0:
                        print("     ⚠️  Aucun point après sigma-clip")
                        result = {
                            'raw': table_tpi,
                            'clean': table_tpi,
                            'ra_clean': ra,
                            'dec_clean': dec,
                            'has_altaz': True
                        }
                        return result
                    
                    # Calcul des vitesses et accélérations si assez de points
                    if np.sum(mask_combined) > self.window_length:
                        vit_ra = savgol_filter(ra[mask_combined], self.window_length, 
                                            self.polyorder, deriv=1, delta=dt)
                        vit_dec = savgol_filter(dec[mask_combined], self.window_length, 
                                                self.polyorder, deriv=1, delta=dt)
                        
                        acc_ra = savgol_filter(ra[mask_combined], self.window_length, 
                                            self.polyorder, deriv=2, delta=dt)
                        acc_dec = savgol_filter(dec[mask_combined], self.window_length, 
                                                self.polyorder, deriv=2, delta=dt)
                        
                        # Filtrage sur les vitesses et accélérations
                        v_norm = np.sqrt(vit_ra**2 + vit_dec**2)
                        a_norm = np.sqrt(acc_ra**2 + acc_dec**2)
                        
                        mask_vel = self.sigma_clip_mask_robust(v_norm, k=3)
                        mask_acc = self.sigma_clip_mask_robust(a_norm, k=3)
                        mask_final = mask_vel & mask_acc
                        
                        # 👉 Vérifier qu'il reste des points après tous les filtres
                        if np.sum(mask_final) == 0:
                            print("     ⚠️  Aucun point après filtrage des vitesses/accélérations")
                            result = {
                                'raw': table_tpi,
                                'clean': table_tpi[mask_combined],
                                'ra_clean': ra[mask_combined],
                                'dec_clean': dec[mask_combined],
                                'has_altaz': True
                            }
                            return result
                        
                        # Appliquer le masque final
                        table_tpi_clipped = table_tpi[mask_combined][mask_final]
                        
                        # Recalculer RA/Dec propres
                        az_clean = table_tpi_clipped["Azimuth"]
                        el_clean = table_tpi_clipped["Elevation"]
                        
                        if 'JD' in table_tpi_clipped.dtype.names:
                            jd_clean = table_tpi_clipped['JD']
                            time_clean = Time(jd_clean, format='jd')
                        else:
                            time_clean = Time.now() + (np.arange(len(table_tpi_clipped)) * u.s)
                        
                        altaz_clean = AltAz(
                            az=az_clean * u.deg,
                            alt=el_clean * u.deg,
                            location=self.location,
                            obstime=time_clean
                        )
                        
                        sky_clean = SkyCoord(altaz_clean)
                        ra_clean = sky_clean.icrs.ra.deg
                        dec_clean = sky_clean.icrs.dec.deg
                        
                        result = {
                            'raw': table_tpi,
                            'clean': table_tpi_clipped,
                            'ra_clean': ra_clean,
                            'dec_clean': dec_clean,
                            'vit_ra': vit_ra[mask_final],
                            'vit_dec': vit_dec[mask_final],
                            'acc_ra': acc_ra[mask_final],
                            'acc_dec': acc_dec[mask_final],
                            'dt': dt,
                            't_cut': t_cut[mask_combined][mask_final],
                            'has_altaz': True
                        }
                    else:
                        print(f"     ⚠️  Pas assez de points pour le filtrage (besoin de >{self.window_length}, trouvé {np.sum(mask_combined)})")
                        result = {
                            'raw': table_tpi,
                            'clean': table_tpi[mask_combined],
                            'ra_clean': ra[mask_combined],
                            'dec_clean': dec[mask_combined],
                            'vit_ra': np.zeros(np.sum(mask_combined)),
                            'vit_dec': np.zeros(np.sum(mask_combined)),
                            'acc_ra': np.zeros(np.sum(mask_combined)),
                            'acc_dec': np.zeros(np.sum(mask_combined)),
                            'dt': dt,
                            't_cut': t_cut[mask_combined],
                            'has_altaz': True
                        }
                else:
                    # Si pas de Alt/Az, retourner juste les données brutes
                    print(f"     ⚠️  Pas de données Alt/Az, affichage brut seulement")
                    result = {
                        'raw': table_tpi,
                        'clean': table_tpi,
                        'has_altaz': False
                    }
                
                return result
                
        except Exception as e:
            print(f"  ❌ Erreur lecture TPI: {e}")
            self.error_files.append({
                'file': filename,
                'target': target,
                'obs_date': obs_date,
                'type': 'TPI',
                'error': str(e)
            })
            return None
    
    def process_image(self, image_file_info, target, obs_date):
        """
        Traite les données image
        """
        file_path = image_file_info['path']
        filename = os.path.basename(file_path)
        
        print(f"  🖼️  Traitement image: {filename}")
        
        try:
            with fits.open(file_path) as hdul:
                # Chercher les données image
                data = None
                for hdu in hdul:
                    if hdu.data is not None and len(hdu.data.shape) >= 2:
                        data = hdu.data
                        break
                
                if data is None:
                    reason = "Pas de données image 2D trouvées"
                    print(f"     ⚠️  Ignoré: {reason}")
                    self.ignored_files.append({
                        'file': filename,
                        'target': target,
                        'obs_date': obs_date,
                        'type': 'IMAGE',
                        'reason': reason
                    })
                    return None
                
                self.processed_files.append({
                    'file': filename,
                    'target': target,
                    'obs_date': obs_date,
                    'type': 'IMAGE'
                })
                
                return {'data': data, 'filename': filename}
                
        except Exception as e:
            print(f"  ❌ Erreur lecture image: {e}")
            self.error_files.append({
                'file': filename,
                'target': target,
                'obs_date': obs_date,
                'type': 'IMAGE',
                'error': str(e)
            })
            return None
    
    @staticmethod
    def sigma_clip_mask_robust(data, k=3):
        """Sigma clipping robuste basé sur la médiane et MAD"""
        data = np.array(data)
        if len(data) < 5:
            return np.ones(len(data), dtype=bool)
        
        median = np.median(data)
        mad = np.median(np.abs(data - median))
        sigma = 1.4826 * mad if mad > 0 else 1.0
        
        mask = np.abs(data - median) < k * sigma
        return mask
    
    def create_directories(self, obs_date, target):
        """Crée l'arborescence des dossiers"""
        base_output_dir = os.path.join(self.base_path, obs_date, self.output_base)
        
        dirs = {
            'base': base_output_dir,
            'plots_unprocessed': os.path.join(base_output_dir, 'plots', target, 'unprocessed'),
            'plots_processed': os.path.join(base_output_dir, 'plots', target, 'processed'),
            'txt_unprocessed': os.path.join(base_output_dir, 'txt', target, 'unprocessed'),
            'txt_processed': os.path.join(base_output_dir, 'txt', target, 'processed')
        }
        
        for dir_path in dirs.values():
            os.makedirs(dir_path, exist_ok=True)
        
        return dirs
    
    def process_observation(self, obs_date):
        """
        Traite toutes les observations d'une date donnée
        """
        obs_dir = os.path.join(self.base_path, obs_date)
        
        if not os.path.exists(obs_dir):
            print(f"❌ Dossier non trouvé: {obs_dir}")
            return
        
        print(f"\n{'='*60}")
        print(f"📅 Traitement de l'observation: {obs_date}")
        print(f"{'='*60}")
        
        files_by_target = self.find_observation_files(obs_dir)
        
        if not files_by_target:
            print("⚠️  Aucun fichier correspondant trouvé")
            return
        
        print(f"\nCibles trouvées: {list(files_by_target.keys())}")
        
        for target, files in files_by_target.items():
            print(f"\n🎯 Cible: {target}")
            print(f"   📊 Spectres annoncés: {len(files['spectrum'])}")
            print(f"   🎯 TPI annoncés: {len(files['tpi'])}")
            print(f"   🖼️  Images annoncées: {len(files['image'])}")
            
            if files['ignored']:
                print(f"   ⚠️  Fichiers ignorés (pattern): {len(files['ignored'])}")
                for ign in files['ignored']:
                    self.ignored_files.append({
                        'file': ign['filename'],
                        'target': target,
                        'obs_date': obs_date,
                        'type': 'PATTERN',
                        'reason': ign['reason']
                    })
            
            # Traiter les spectres
            for spec_file in files['spectrum']:
                dirs = self.create_directories(obs_date, target)
                spectrum_results = self.process_spectrum_unprocessed(spec_file, target, obs_date)
                
                # 👉 IMPORTANT: Ne sauvegarder que si on a des résultats
                if spectrum_results is not None:
                    self.save_plots(spectrum_results, None, None, target, obs_date, dirs, spec_file['time'])
                    self.save_text_files(spectrum_results, None, None, target, obs_date, dirs, spec_file['time'])
                    print(f"  ✅ Spectre sauvegardé")
            
            # Traiter les TPI
            for tpi_file in files['tpi']:
                dirs = self.create_directories(obs_date, target)
                tpi_results = self.process_tpi(tpi_file, target, obs_date)
                
                # 👉 IMPORTANT: Ne sauvegarder que si on a des résultats
                if tpi_results is not None:
                    self.save_plots(None, tpi_results, None, target, obs_date, dirs, tpi_file['time'])
                    self.save_text_files(None, tpi_results, None, target, obs_date, dirs, tpi_file['time'])
                    print(f"  ✅ TPI sauvegardé")
            
            # Traiter les images
            for img_file in files['image']:
                dirs = self.create_directories(obs_date, target)
                image_results = self.process_image(img_file, target, obs_date)
                
                # 👉 IMPORTANT: Ne sauvegarder que si on a des résultats
                if image_results is not None:
                    self.save_plots(None, None, image_results, target, obs_date, dirs, img_file['time'])
                    print(f"  ✅ Image sauvegardée")
    
    def save_plots(self, spectrum_results, tpi_results, image_results, target, obs_date, dirs, file_time):
        """
        Sauvegarde tous les plots (processed et unprocessed)
        Ne fait rien si tous les résultats sont None
        Pour les spectres, on ignore les données de position
        """
        # 👉 Vérifier qu'au moins un type de données est présent
        if spectrum_results is None and tpi_results is None and image_results is None:
            print(f"     ⚠️  Rien à plotter pour {target} {file_time}")
            return
        
        timestamp = file_time
        
        # ========== PLOTS UNPROCESSED ==========
        
        # Plot spectre brut (unprocessed) - seulement si on a des résultats
        if spectrum_results is not None:
            plt.figure(figsize=(12, 6))
            
            if 'raw_data' in spectrum_results:
                data = spectrum_results['raw_data']
                
                if hasattr(data, 'dtype') and data.dtype.names is not None:
                    cols = data.dtype.names
                    
                    # On ne garde que les données de spectre pures
                    # LEFT_POL et RIGHT_POL sont les colonnes de spectre
                    if 'LEFT_POL' in cols and 'RIGHT_POL' in cols:
                        if len(data) > 0:
                            # Chercher les lignes ON et OFF si disponibles
                            if 'STATUS' in cols:
                                on_rows = data[data['STATUS'] == 'on']
                                off_rows = data[data['STATUS'] == 'off']
                                
                                if len(on_rows) > 0:
                                    on_row = on_rows[0]
                                    plt.plot(on_row['LEFT_POL'], label='ON LEFT', alpha=0.7, linewidth=1)
                                    plt.plot(on_row['RIGHT_POL'], label='ON RIGHT', alpha=0.7, linewidth=1)
                                
                                if len(off_rows) > 0:
                                    off_row = off_rows[0]
                                    plt.plot(off_row['LEFT_POL'], label='OFF LEFT', alpha=0.7, linewidth=1, linestyle='--')
                                    plt.plot(off_row['RIGHT_POL'], label='OFF RIGHT', alpha=0.7, linewidth=1, linestyle='--')
                            else:
                                # Sinon afficher toutes les polarisations disponibles
                                for i, row in enumerate(data[:min(2, len(data))]):
                                    plt.plot(row['LEFT_POL'], label=f'LEFT row{i}', alpha=0.7, linewidth=1)
                                    plt.plot(row['RIGHT_POL'], label=f'RIGHT row{i}', alpha=0.7, linewidth=1)
                            
                            plt.ylabel("Signal (ADU)")
                            plt.title(f"Spectre - {target} ({obs_date} {timestamp})")
                            plt.legend(fontsize='small')
                            plt.grid(True, alpha=0.3)
                            plt.xlabel("Canal")
                        
                        else:
                            print(f"     ⚠️  Pas de données dans le spectre")
                    
                    # Si LEFT_POL/RIGHT_POL ne sont pas disponibles, chercher d'autres colonnes de spectre
                    elif any(col in cols for col in ['DATA', 'SPECTRUM', 'AMP']):
                        # Chercher la première colonne de spectre disponible
                        for col in ['DATA', 'SPECTRUM', 'AMP', 'POWER']:
                            if col in cols:
                                plt.plot(data[col], label=col, alpha=0.7, linewidth=1)
                                break
                        plt.ylabel("Signal")
                        plt.title(f"Spectre - {target} ({obs_date} {timestamp})")
                        plt.legend()
                        plt.grid(True, alpha=0.3)
                        plt.xlabel("Canal")
                    
                    else:
                        # Si vraiment on ne trouve rien, afficher les premières colonnes numériques
                        # mais en excluant les colonnes de position évidentes
                        exclude_cols = ['Azimuth', 'Elevation', 'JD', 'Az_Offset', 'El_Offset', 
                                       'Gal_Long', 'Gal_Lat', 'MARKER']
                        
                        plotted = 0
                        for col in cols[:10]:
                            if col not in exclude_cols and plotted < 4:
                                try:
                                    if np.issubdtype(data[col].dtype, np.number):
                                        plt.plot(data[col], label=col, alpha=0.7, linewidth=1)
                                        plotted += 1
                                except:
                                    pass
                        
                        if plotted > 0:
                            plt.ylabel("Valeur")
                            plt.title(f"Données spectre - {target} ({obs_date} {timestamp})")
                            plt.legend(fontsize='small')
                            plt.grid(True, alpha=0.3)
                            plt.xlabel("Canal")
                        else:
                            print(f"     ⚠️  Aucune donnée de spectre trouvée")
                else:
                    # Si c'est un simple array
                    plt.plot(data.flatten()[:1024], linewidth=1)
                    plt.xlabel("Canal")
                    plt.ylabel("Valeur")
                    plt.title(f"Spectre - {target} ({obs_date} {timestamp})")
            
            plt.tight_layout()
            plot_path = os.path.join(dirs['plots_unprocessed'], f"{target}_spectrum_{timestamp}.png")
            plt.savefig(plot_path, dpi=150)
            plt.close()
            print(f"     💾 Spectre sauvegardé: {os.path.basename(plot_path)}")
        
        # Plot TPI brut (unprocessed) - seulement si on a des résultats
        if tpi_results is not None and 'raw' in tpi_results:
            plt.figure(figsize=(12, 8))
            
            raw_data = tpi_results['raw']
            
            # Si c'est une table structurée
            if hasattr(raw_data, 'dtype') and raw_data.dtype.names is not None:
                # Chercher les colonnes BBC ou canaux
                bbc_columns = [col for col in raw_data.dtype.names 
                              if any(x in col for x in ['BBC', 'CH', 'AMP', 'POWER', 'DB'])]
                
                if bbc_columns:
                    for i, col in enumerate(bbc_columns[:8]):  # Max 8 courbes pour lisibilité
                        plt.plot(raw_data[col], label=col, alpha=0.7, linewidth=1)
                    
                    plt.xlabel("Index")
                    plt.ylabel("Signal (ADU)")
                    plt.title(f"BBCs - {target} ({obs_date} {timestamp})")
                    plt.legend(loc='upper right', ncol=2, fontsize='small')
                else:
                    # Si pas de BBC, plot des premières colonnes numériques
                    numeric_cols = []
                    for col in raw_data.dtype.names[:10]:
                        try:
                            if np.issubdtype(raw_data[col].dtype, np.number):
                                numeric_cols.append(col)
                                plt.plot(raw_data[col], label=col, alpha=0.7, linewidth=1)
                        except:
                            pass
                    
                    plt.xlabel("Index")
                    plt.ylabel("Valeur")
                    plt.title(f"Données TPI - {target} ({obs_date} {timestamp})")
                    if numeric_cols:
                        plt.legend(loc='upper right', fontsize='small')
            else:
                # Si c'est un simple array
                plt.plot(raw_data.flatten()[:1000], linewidth=1)
                plt.xlabel("Index")
                plt.ylabel("Valeur")
                plt.title(f"TPI - {target} ({obs_date} {timestamp})")
            
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plot_path = os.path.join(dirs['plots_unprocessed'], f"{target}_tpi_{timestamp}.png")
            plt.savefig(plot_path, dpi=150)
            plt.close()
            print(f"     💾 TPI sauvegardé: {os.path.basename(plot_path)}")
        
        # Plot image si disponible - seulement si on a des résultats
        if image_results is not None and 'data' in image_results:
            plt.figure(figsize=(10, 8))
            data = image_results['data']
            
            # Si l'image est 2D
            if len(data.shape) == 2:
                plt.imshow(data, cmap='viridis', origin='lower', aspect='auto')
                plt.colorbar(label='Intensité')
                plt.title(f"Image - {target} ({obs_date} {timestamp})")
                plt.xlabel("X")
                plt.ylabel("Y")
            else:
                # Si 1D, plot simple
                plt.plot(data.flatten())
                plt.title(f"Image - {target} ({obs_date} {timestamp})")
                plt.xlabel("Index")
                plt.ylabel("Valeur")
            
            plt.tight_layout()
            plot_path = os.path.join(dirs['plots_unprocessed'], f"{target}_image_{timestamp}.png")
            plt.savefig(plot_path, dpi=150)
            plt.close()
            print(f"     💾 Image sauvegardée: {os.path.basename(plot_path)}")
        
        # ========== PLOTS PROCESSED ==========
        # Uniquement pour les TPI qui ont été traités avec succès
        if tpi_results is not None and 'clean' in tpi_results and tpi_results.get('has_altaz', False):
            
            if 'ra_clean' in tpi_results and len(tpi_results['ra_clean']) > 0:
                fig, axes = plt.subplots(2, 2, figsize=(14, 10))
                
                # RA vs Dec
                axes[0, 0].scatter(tpi_results['ra_clean'], tpi_results['dec_clean'], 
                                  s=5, alpha=0.5, c='blue')
                axes[0, 0].set_xlabel("RA (deg)")
                axes[0, 0].set_ylabel("Dec (deg)")
                axes[0, 0].set_title(f"RA/Dec - {target}")
                axes[0, 0].invert_xaxis()
                axes[0, 0].grid(True, alpha=0.3)
                
                # Vitesses
                if 'vit_ra' in tpi_results and len(tpi_results['vit_ra']) > 0:
                    axes[0, 1].scatter(tpi_results['vit_ra'], tpi_results['vit_dec'], 
                                      s=5, alpha=0.5, c='red')
                    axes[0, 1].set_xlabel("Vitesse RA (deg/s)")
                    axes[0, 1].set_ylabel("Vitesse Dec (deg/s)")
                    axes[0, 1].set_title("Vitesses RA/Dec")
                    axes[0, 1].grid(True, alpha=0.3)
                
                # Accélérations
                if 'acc_ra' in tpi_results and len(tpi_results['acc_ra']) > 0:
                    axes[1, 0].scatter(tpi_results['acc_ra'], tpi_results['acc_dec'], 
                                      s=5, alpha=0.5, c='green')
                    axes[1, 0].set_xlabel("Accélération RA (deg/s²)")
                    axes[1, 0].set_ylabel("Accélération Dec (deg/s²)")
                    axes[1, 0].set_title("Accélérations RA/Dec")
                    axes[1, 0].grid(True, alpha=0.3)
                
                # BBCs nettoyées
                if 'clean' in tpi_results and hasattr(tpi_results['clean'], 'dtype'):
                    clean_data = tpi_results['clean']
                    if clean_data.dtype.names is not None:
                        bbc_columns = [col for col in clean_data.dtype.names 
                                      if any(x in col for x in ['BBC', 'CH', 'AMP', 'POWER'])]
                        
                        for i, col in enumerate(bbc_columns[:8]):
                            axes[1, 1].plot(clean_data[col], 
                                           label=col, alpha=0.7, linewidth=1)
                        
                        axes[1, 1].set_xlabel("Index")
                        axes[1, 1].set_ylabel("Signal (ADU)")
                        axes[1, 1].set_title("BBCs après filtrage")
                        if bbc_columns:
                            axes[1, 1].legend(loc='upper right', ncol=2, fontsize='small')
                        axes[1, 1].grid(True, alpha=0.3)
                
                plt.suptitle(f"{target} - {obs_date} {timestamp}")
                plt.tight_layout()
                plot_path = os.path.join(dirs['plots_processed'], f"{target}_tpi_processed_{timestamp}.png")
                plt.savefig(plot_path, dpi=150)
                plt.close()
                print(f"     💾 TPI traité sauvegardé: {os.path.basename(plot_path)}")
    
    def save_text_files(self, spectrum_results, tpi_results, image_results, target, obs_date, dirs, file_time):
        """Sauvegarde les fichiers texte"""
        timestamp = file_time
        
        if spectrum_results and 'raw_data' in spectrum_results:
            txt_path = os.path.join(dirs['txt_unprocessed'], f"{target}_spectrum_{timestamp}.txt")
            
            with open(txt_path, 'w') as f:
                f.write(f"# Spectre - {target} - {obs_date} {timestamp}\n")
                data = spectrum_results['raw_data']
                
                if hasattr(data, 'dtype') and data.dtype.names:
                    col_names = list(data.dtype.names)
                    f.write("# " + "\t".join(col_names) + "\n")
                    
                    for i in range(min(len(data), 1000)):
                        row = data[i]
                        values = [str(row[col]) for col in col_names]
                        f.write("\t".join(values) + "\n")
        
        if tpi_results and 'clean' in tpi_results:
            clean_data = tpi_results['clean']
            
            # 👉 Vérifier que clean_data n'est pas vide
            if len(clean_data) == 0:
                print(f"     ⚠️  Données TPI vides pour {target} {timestamp}, fichier texte non sauvegardé")
                return
            
            txt_path = os.path.join(dirs['txt_processed'], f"{target}_tpi_processed_{timestamp}.txt")
            
            with open(txt_path, 'w') as f:
                f.write(f"# Données TPI traitées - {target} - {obs_date} {timestamp}\n")
                f.write(f"# Généré le: {datetime.now().isoformat()}\n")
                
                if hasattr(clean_data, 'dtype') and clean_data.dtype.names is not None:
                    col_names = list(clean_data.dtype.names)
                    
                    # Ajouter RA/Dec si disponibles
                    extra_cols = []
                    if 'ra_clean' in tpi_results:
                        extra_cols = ['RA_clean', 'Dec_clean']
                    
                    f.write("# " + "\t".join(col_names + extra_cols) + "\n")
                    
                    # 👉 Correction: éviter la division par zéro
                    max_rows = min(10000, len(clean_data))
                    if max_rows > 0:
                        step = max(1, len(clean_data) // max_rows)
                        
                        for i in range(0, len(clean_data), step):
                            row = clean_data[i]
                            values = [str(row[col]) for col in col_names]
                            
                            if 'ra_clean' in tpi_results and i < len(tpi_results['ra_clean']):
                                values.append(str(tpi_results['ra_clean'][i]))
                                values.append(str(tpi_results['dec_clean'][i]))
                            
                            f.write("\t".join(values) + "\n")
                    else:
                        # Si max_rows est 0 (ne devrait pas arriver après la vérification)
                        f.write("# Aucune donnée valide\n")
                else:
                    f.write("# Index\tValeur\n")
                    flat_data = clean_data.flatten()
                    for i, val in enumerate(flat_data[:10000]):
                        f.write(f"{i}\t{val}\n")
    
    def generate_report(self):
        """Génère un rapport final avec tous les fichiers ignorés"""
        print("\n" + "="*80)
        print("📊 RAPPORT FINAL")
        print("="*80)
        
        print(f"\n✅ Fichiers traités avec succès: {len(self.processed_files)}")
        for pf in self.processed_files:
            print(f"   - {pf['obs_date']} | {pf['target']} | {pf['type']:8} | {pf['file']}")
        
        print(f"\n⚠️  Fichiers ignorés: {len(self.ignored_files)}")
        if self.ignored_files:
            # Grouper par raison
            reasons = {}
            for ign in self.ignored_files:
                reason = ign['reason']
                if reason not in reasons:
                    reasons[reason] = []
                reasons[reason].append(ign)
            
            for reason, files in reasons.items():
                print(f"\n   📌 {reason} ({len(files)} fichiers):")
                for f in files[:10]:  # Afficher les 10 premiers
                    print(f"      - {f['obs_date']} | {f['target']} | {f['file']}")
                if len(files) > 10:
                    print(f"      ... et {len(files)-10} autres")
        
        print(f"\n❌ Fichiers en erreur: {len(self.error_files)}")
        for err in self.error_files:
            print(f"   - {err['obs_date']} | {err['target']} | {err['file']}: {err['error']}")
        
        # Sauvegarder le rapport dans un fichier
        report_path = os.path.join(self.base_path, "pipeline_report.txt")
        with open(report_path, 'w') as f:
            f.write("RAPPORT DU PIPELINE GRAD-300\n")
            f.write(f"Généré le: {datetime.now().isoformat()}\n\n")
            
            f.write(f"Traités: {len(self.processed_files)}\n")
            for pf in self.processed_files:
                f.write(f"  [OK] {pf['obs_date']} {pf['target']} {pf['type']} {pf['file']}\n")
            
            f.write(f"\nIgnorés: {len(self.ignored_files)}\n")
            for ign in self.ignored_files:
                f.write(f"  [IGN] {ign['obs_date']} {ign['target']} {ign['reason']} {ign['file']}\n")
            
            f.write(f"\nErreurs: {len(self.error_files)}\n")
            for err in self.error_files:
                f.write(f"  [ERR] {err['obs_date']} {err['target']} {err['error']} {err['file']}\n")
        
        print(f"\n📄 Rapport détaillé sauvegardé dans: {report_path}")
    
    def run(self):
        """Lance le pipeline complet"""
        obs_dates = []
        for item in os.listdir(self.base_path):
            item_path = os.path.join(self.base_path, item)
            if os.path.isdir(item_path) and re.match(r'\d{8}', item):
                fits_dir = os.path.join(item_path, "FITS")
                if os.path.exists(fits_dir):
                    obs_dates.append(item)
        
        obs_dates.sort()
        
        if not obs_dates:
            print("❌ Aucun dossier d'observation trouvé")
            return
        
        print(f"🔍 Dossiers trouvés: {obs_dates}")
        
        for obs_date in obs_dates:
            self.process_observation(obs_date)
        
        self.generate_report()

# ==================== UTILISATION ====================

if __name__ == "__main__":
    BASE_PATH = "./GRAD-300/GRAD-300"
    OUTPUT_NAME = "output_nametarget"
    
    print(f"🔍 Recherche dans: {BASE_PATH}")
    pipeline = RadioPipelineGRAD300(BASE_PATH, OUTPUT_NAME)
    pipeline.run()

🔍 Recherche dans: ./GRAD-300/GRAD-300
🔍 Dossiers trouvés: ['20250130', '20250131', '20250204', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213']

📅 Traitement de l'observation: 20250130
  📁 7 fichiers trouvés dans ./GRAD-300/GRAD-300/20250130/FITS

Cibles trouvées: ['AZEL', 'TAUA', 'MOON', 'CYGA', 'CASA']

🎯 Cible: AZEL
   📊 Spectres annoncés: 1
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 1
  📊 Vérification spectre: 20250130-152518_SPECTRUM-PROJ01-AZEL_03#_01#.fits
     ⚠️  Ignoré: Pas un vrai spectre (manque LEFT_POL/RIGHT_POL/STATUS ou contient Az/El)
  🖼️  Traitement image: 20250130-193656_IMAGE-PROJ01-AZEL_01#_01#.fits


    Header size is not multiple of 2880: 148140
There may be extra bytes after the last HDU or the file is corrupted. [astropy.io.fits.hdu.hdulist]


     💾 Image sauvegardée: AZEL_image_193656.png
  ✅ Image sauvegardée

🎯 Cible: TAUA
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 1
  🖼️  Traitement image: 20250130-192321_IMAGE-PROJ01-TAUA_01#_01#.fits


     💾 Image sauvegardée: TAUA_image_192321.png
  ✅ Image sauvegardée

🎯 Cible: MOON
   📊 Spectres annoncés: 1
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 0
  📊 Vérification spectre: 20250130-152816_SPECTRUM-PROJ01-MOON_04#_01#.fits
     ⚠️  Ignoré: Pas un vrai spectre (manque LEFT_POL/RIGHT_POL/STATUS ou contient Az/El)

🎯 Cible: CYGA
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 2
  🖼️  Traitement image: 20250130-154009_IMAGE-PROJ01-CYGA_01#_01#.fits


     💾 Image sauvegardée: CYGA_image_154009.png
  ✅ Image sauvegardée
  🖼️  Traitement image: 20250130-154309_IMAGE-PROJ01-CYGA_01#_01#.fits
     💾 Image sauvegardée: CYGA_image_154309.png
  ✅ Image sauvegardée

🎯 Cible: CASA
   📊 Spectres annoncés: 1
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 0
  📊 Vérification spectre: 20250130-153331_SPECTRUM-PROJ01-CASA_05#_01#.fits
     ⚠️  Ignoré: Pas un vrai spectre (manque LEFT_POL/RIGHT_POL/STATUS ou contient Az/El)

📅 Traitement de l'observation: 20250131
  📁 2 fichiers trouvés dans ./GRAD-300/GRAD-300/20250131/FITS

Cibles trouvées: ['AZEL', 'MOON']

🎯 Cible: AZEL
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 1
  🖼️  Traitement image: 20250131-071839_IMAGE-PROJ01-AZEL_01#_01#.fits
     💾 Image sauvegardée: AZEL_image_071839.png
  ✅ Image sauvegardée

🎯 Cible: MOON
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 0
   🖼️  Images annoncées: 1
  🖼️  Traitement image: 20250131-150755_IMAGE-PROJ01-MOON_01#_01#.fits
  ❌ Er

...

     💾 TPI sauvegardé: SUN_tpi_123607.png
     💾 TPI traité sauvegardé: SUN_tpi_processed_123607.png
  ✅ TPI sauvegardé
  🎯 Traitement TPI: 20250213-095939_TPI-PROJ01-SUN_02#_01#.fits


     💾 TPI sauvegardé: SUN_tpi_095939.png
     💾 TPI traité sauvegardé: SUN_tpi_processed_095939.png
  ✅ TPI sauvegardé
  🖼️  Traitement image: 20250213-123605_IMAGE-PROJ01-SUN_01#_01#.fits
     💾 Image sauvegardée: SUN_image_123605.png
  ✅ Image sauvegardée
  🖼️  Traitement image: 20250213-095937_IMAGE-PROJ01-SUN_01#_01#.fits
     💾 Image sauvegardée: SUN_image_095937.png
  ✅ Image sauvegardée

🎯 Cible: CYGA
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 1
   🖼️  Images annoncées: 1
  🎯 Traitement TPI: 20250213-074850_TPI-PROJ01-CYGA_01#_01#.fits


  ❌ Erreur lecture TPI: Latitude angle(s) must be within -90 deg <= angle <= 90 deg, got 60.67 deg <= angle <= 90.01 deg
  🖼️  Traitement image: 20250213-074846_IMAGE-PROJ01-CYGA_01#_01#.fits
     💾 Image sauvegardée: CYGA_image_074846.png
  ✅ Image sauvegardée

🎯 Cible: CASA
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 1
   🖼️  Images annoncées: 1
  🎯 Traitement TPI: 20250213-145925_TPI-PROJ01-CASA_04#_01#.fits


     💾 TPI sauvegardé: CASA_tpi_145925.png
     💾 TPI traité sauvegardé: CASA_tpi_processed_145925.png
  ✅ TPI sauvegardé
  🖼️  Traitement image: 20250213-145921_IMAGE-PROJ01-CASA_01#_01#.fits
     💾 Image sauvegardée: CASA_image_145921.png
  ✅ Image sauvegardée

🎯 Cible: TAUA
   📊 Spectres annoncés: 0
   🎯 TPI annoncés: 1
   🖼️  Images annoncées: 1
  🎯 Traitement TPI: 20250213-184853_TPI-PROJ01-TAUA_05#_01#.fits
     ⚠️  Pas de colonne JD, pas de filtrage temporel
     ⚠️  Aucune donnée après filtrage temporel
     💾 TPI sauvegardé: TAUA_tpi_184853.png
     ⚠️  Données TPI vides pour TAUA 184853, fichier texte non sauvegardé
  ✅ TPI sauvegardé
  🖼️  Traitement image: 20250213-184848_IMAGE-PROJ01-TAUA_01#_01#.fits
  ❌ Erreur lecture image: Empty or corrupt FITS file

📊 RAPPORT FINAL

✅ Fichiers traités avec succès: 79
   - 20250130 | AZEL | IMAGE    | 20250130-193656_IMAGE-PROJ01-AZEL_01#_01#.fits
   - 20250130 | TAUA | IMAGE    | 20250130-192321_IMAGE-PROJ01-TAUA_01#_01#.fits
   - 202

    Header size is not multiple of 2880: 37944
There may be extra bytes after the last HDU or the file is corrupted. [astropy.io.fits.hdu.hdulist]
